# Inverse problem: which (N, a) give a target R_eff?

The forward sweeps (`effective_resistance_vs_N_a.ipynb`) map (N, a) -> R_eff. This notebook goes
the other way: **given a target R_eff, which (N, a) pairs produce it?**

The short answer is: it's not a unique pair. R_eff appears to collapse (approximately) onto the
combined variable $x = N \cdot a^2$ (the stick-percolation excluded-area parameter), so a whole
*curve* of (N, a) combinations gives roughly the same R_eff -- more, shorter sticks behave like
fewer, longer ones. That assumption is checked, not just assumed (section 3.1).

Approach:
1. Pool every (N, a) -> R_eff sample from the forward sweeps already run (`experiments/exp_*/effective_resistance_sweep_summary*.json`).
2. Fit the empirical power law $R_{eff} \approx A \cdot (N a^2)^{s}$ on that pooled data.
3. **Cross-check the N*a^2 assumption**: fit the more general, unconstrained $R_{eff} \approx A' \cdot N^p \cdot a^q$ and compare -- if $p \not\approx 2q$, the N*a^2 collapse doesn't actually hold and the general fit is used instead.
4. Invert whichever fit wins: for a target R_eff, solve for the (N, a) solution curve.
5. **Validate with new experiments**: for a few target R_eff values, pick candidate (N, a) points off the predicted curve, actually generate percolating networks at those points, and check how close the real R_eff lands to the target.
6. Save the fits and the validation experiment results (as always, via `ExperimentManager`).

---
## 1. Setup and imports

In [1]:
import sys
import os
import glob
import json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    experiments_dir = "/content/drive/MyDrive/ColabProject/experiments"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    experiments_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "experiments"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")
print(f"Experiments dir: {experiments_dir}")

Environment: Local
Utils path: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils
Experiments dir: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/experiments


In [2]:
from pipeline_utils import generate_percolating_network, distance_dependent_impedance, stable_seed
from experiment_manager import setup_experiment

R0 = 1000.0  # same distance-dependent resistance scale used throughout the project

print("Imports OK")

Imports OK


---
## 2. Pool every forward-sweep sample seen so far

Scans `experiments/exp_*/` for any `effective_resistance_sweep_summary*.json` (both the small
12-pair sweep and the larger adaptive-grid sweep produce this format) and merges every
(N, a) -> R_eff_values record it finds. Re-running this notebook later, after more sweeps have
been run, automatically picks up the extra data -- nothing here is hardcoded to one experiment.

In [3]:
import re

# Only pool sweeps produced AFTER the merge-geometry fix of 2026-08-31. Before it,
# pipeline_utils.distance_dependent_impedance measured edge lengths on the MERGED
# geometry, where merge_boundary_nodes parks each terminal at the average position of
# the face it replaces, which inflated every whole-face R_eff by an amount that grows
# with density (a few percent at N=20, over 30% at N=100). Those runs are still on disk
# under lower experiment numbers, and pooling them with the corrected ones mixes two
# different physical models: fitted on stale data alone the N exponent is -1.28, on
# corrected data alone it is -1.51, and the mixture lands in between and drifts as more
# runs accumulate. Experiment 843 is the first re-run after the fix.
MERGE_FIX_FIRST_EXPERIMENT = 843

all_paths = sorted(glob.glob(os.path.join(experiments_dir, "exp_*", "effective_resistance_sweep_summary*.json")))
# Since 2026-09-15 the generator clips sticks at the box edge instead of rejecting them
# (see utils/point_generator.py) and every sweep summary records which rule it used in
# config["boundary"]. Only 'clip' summaries are pooled; files without the key predate the
# change and are skipped, whatever their experiment number.
BOUNDARY_RULE = "clip"

def _boundary_of(path):
    with open(path) as f:
        return json.load(f).get("config", {}).get("boundary")

summary_paths = [p for p in all_paths
                 if int(re.search(r"exp_(\d+)", p).group(1)) >= MERGE_FIX_FIRST_EXPERIMENT
                 and _boundary_of(p) == BOUNDARY_RULE]
skipped = len(all_paths) - len(summary_paths)

if not summary_paths:
    raise RuntimeError(
        f"No sweep summary from exp_{MERGE_FIX_FIRST_EXPERIMENT} or later found in "
        f"{experiments_dir}. Every sweep on this machine predates the merge-geometry fix, "
        f"so the fit would be built on the biased whole-face R_eff. Run "
        f"effective_resistance_vs_N_a.ipynb first.")

print(f"Found {len(all_paths)} sweep summary file(s), pooling {len(summary_paths)} "
      f"with boundary={BOUNDARY_RULE!r} from exp_{MERGE_FIX_FIRST_EXPERIMENT} onward ({skipped} skipped):")
for p in summary_paths:
    print(f"  {p}")

# Every draw is seeded from (N, a) via stable_seed(), so re-running a sweep reproduces
# its trials byte for byte. experiments/ holds several such re-runs of the same grids
# (exp_843/854/856/862 are one baseline sweep, exp_844/857 one extended sweep); pooling
# every file would count the same trials several times and weight the grids by how
# often they were re-run rather than by how many trials they hold. A cell whose
# R_eff_values vector has already been pooled is therefore skipped. Cells at the same
# (N, a) with DIFFERENT values (the baseline and extended grids overlap at a few points,
# drawn with different seeds) are genuinely independent and are kept.
merged = defaultdict(list)  # (N, a) -> list of individual R_eff samples (ohm)
seen_cells = set()
n_dup_cells = 0
for p in summary_paths:
    with open(p) as f:
        summary = json.load(f)
    for rec in summary.get("sweep", []):
        vals = rec.get("R_eff_values") or []
        if not vals:
            continue
        key = (rec["N"], round(rec["a"], 3))
        # re-runs on another machine reproduce a seeded cell only to ~1e-12 (exp_883 on the
        # laptop vs exp_870 on the cluster), so cells are compared at 9 significant digits
        cell_id = (key, tuple(float(f"{v:.9g}") for v in vals))
        if cell_id in seen_cells:
            n_dup_cells += 1
            continue
        seen_cells.add(cell_id)
        merged[key].extend(vals)
print(f"Skipped {n_dup_cells} duplicate cell(s) from repeated seeded re-runs")

Ns_pts = np.array([k[0] for k in merged])
as_pts = np.array([k[1] for k in merged])
R_mean_pts = np.array([np.mean(v) for v in merged.values()])
n_pts = len(merged)
n_total_trials = sum(len(v) for v in merged.values())

print(f"\n{n_pts} distinct (N, a) points pooled, {n_total_trials} unique percolating trials")
print(f"R_eff range: {R_mean_pts.min():.1f} - {R_mean_pts.max():.1f} ohm")
print(f"N range: {Ns_pts.min()} - {Ns_pts.max()}    a range: {as_pts.min():.1f} - {as_pts.max():.1f}")

Found 17 sweep summary file(s), pooling 3 with boundary='clip' from exp_843 onward (14 skipped):
  /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/experiments/exp_870/effective_resistance_sweep_summary.json
  /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/experiments/exp_871/effective_resistance_sweep_summary_ext.json
  /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/experiments/exp_883/effective_resistance_sweep_summary.json
Skipped 0 duplicate cell(s) from repeated seeded re-runs

101 distinct (N, a) points pooled, 3099 unique percolating trials
R_eff range: 24.8 - 1273.5 ohm
N range: 20 - 150    a range: 0.2 - 1.1


---
## 3. Fit the collapse: R_eff vs N*a^2

Simple log-log linear regression (`numpy.polyfit`, no scipy needed) on every individual
percolating trial (not just the per-pair means), so pairs with more trials get proportionally
more weight in the fit.

In [4]:
Ns_flat, as_flat, R_flat = [], [], []
for (N, a), vals in merged.items():
    Ns_flat.extend([N] * len(vals))
    as_flat.extend([a] * len(vals))
    R_flat.extend(vals)
Ns_flat = np.array(Ns_flat)
as_flat = np.array(as_flat)
R_flat = np.array(R_flat)

x_flat = Ns_flat * as_flat**2
logx = np.log10(x_flat)
logR = np.log10(R_flat)

slope, intercept = np.polyfit(logx, logR, 1)
A = 10**intercept

pred = slope * logx + intercept
ss_res = np.sum((logR - pred)**2)
ss_tot = np.sum((logR - logR.mean())**2)
r2 = 1 - ss_res / ss_tot

print(f"Fit:  R_eff  ~=  {A:.2f} * (N * a^2) ^ ({slope:.3f})")
print(f"R^2 (log-log) = {r2:.3f}   n = {len(R_flat)} individual trials")

Fit:  R_eff  ~=  4898.18 * (N * a^2) ^ (-1.277)
R^2 (log-log) = 0.850   n = 3099 individual trials


In [5]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(x_flat, R_flat, c=as_flat, cmap='plasma', s=18, alpha=0.6, edgecolor='none')
x_line = np.logspace(np.log10(x_flat.min()), np.log10(x_flat.max()), 100)
ax.plot(x_line, A * x_line**slope, 'k--', linewidth=2, label=f'fit: R = {A:.1f} * x^{slope:.2f}  (R^2={r2:.3f})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('N * a^2'); ax.set_ylabel('R_eff (ohm)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
fig.colorbar(sc, ax=ax, label='a')
plt.tight_layout()
plt.show()
_collapse_fig = fig  # saved to the experiment directory in section 6, with the other fit figures


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/559901042.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### 3.1 Cross-check: does N and a really only enter through N*a^2?

The fit above forces the exponents to 1 on N and 2 on a (i.e. R_eff depends on N and a only
through the combined variable N*a^2). That combination is well-justified for the *percolation
threshold* (it's the mean-crossings-per-stick excluded-area parameter from continuum
stick-percolation theory), but nothing guarantees R_eff itself -- away from the threshold --
collapses onto exactly the same variable.

So here we fit the more general, unconstrained form $R_{eff} \approx A' \cdot N^p \cdot a^q$ and
check whether $p$ ends up close to $2q$. If it does, the N*a^2 collapse is a good description of
this data; if not, this more general fit should be trusted instead, and is used for the
inversion in section 4.

In [6]:
X = np.column_stack([np.ones_like(logx), np.log10(Ns_flat), np.log10(as_flat)])
coef, *_ = np.linalg.lstsq(X, logR, rcond=None)
log_Ap, p_exp, q_exp = coef
Ap = 10**log_Ap

pred_general = X @ coef
ss_res_g = np.sum((logR - pred_general) ** 2)
r2_general = 1 - ss_res_g / ss_tot

print(f"General fit:   R_eff ~= {Ap:.2f} * N^{p_exp:.3f} * a^{q_exp:.3f}   [R^2 = {r2_general:.3f}]")
print(f"N*a^2 fit:     R_eff ~= {A:.2f} * (N*a^2)^{slope:.3f}             [R^2 = {r2:.3f}]")
print(f"\np = {p_exp:.3f}   2*q = {2 * q_exp:.3f}   (ratio p/(2q) = {p_exp / (2 * q_exp):.3f}; "
      f"close to 1.0 means the N*a^2 collapse is well justified)")

use_general = r2_general > r2 + 0.01  # only switch models if the improvement isn't just noise
print(f"\n-> Using the {'GENERAL' if use_general else 'N*a^2'} fit for the inversion below "
      f"({'higher R^2 by > 0.01' if use_general else 'the two fits are close enough that the simpler model is preferred'}).")

def predict_N(R_target, a):
    """N required to hit R_target at a given a, per whichever fit scored better above."""
    if use_general:
        return 10 ** ((np.log10(R_target / Ap) - q_exp * np.log10(a)) / p_exp)
    x_t = 10 ** ((np.log10(R_target) - intercept) / slope)
    return x_t / a**2

General fit:   R_eff ~= 28259.46 * N^-1.695 * a^-2.460   [R^2 = 0.904]
N*a^2 fit:     R_eff ~= 4898.18 * (N*a^2)^-1.277             [R^2 = 0.850]

p = -1.695   2*q = -4.919   (ratio p/(2q) = 0.345; close to 1.0 means the N*a^2 collapse is well justified)

-> Using the GENERAL fit for the inversion below (higher R^2 by > 0.01).


---
### 3.2 The percolation critical form, for comparison with theory

Percolation theory does not predict a pure power law in `N`. Near the threshold the
conductivity of a 2D percolating system goes as `(n - n_c)^t` with the universal exponent
`t ~ 1.31` (Grassberger 1999), and for sticks the threshold is set by the excluded-area
variable, `N_c(a) = c / a^2` with `c ~ 5.64` for an infinite system (Pike & Seager 1974).
In the wire-limited model used here every segment resistance also scales with the stick
length, so an `a^q` prefactor is expected on top. The natural form to compare with is
therefore

```
R_eff = A * a^q * (N*a^2 - c)^(-t)
```

fitted by nonlinear least squares in log space over the same individual trials, with `c`
free (this box is only 1/a = 1 to 5 stick lengths across, and percolating draws exist well
below 5.64, so the effective threshold of the finite geometry is not the infinite-system
one) and, for reference, with `c` pinned at 5.64 on the trials above it. Uncertainties
are a cluster bootstrap over the `(N, a)` cells, so trials from the same cell are never
treated as independent resamples. The dense wire-limited limit, in which the sheet
conductance is proportional to the total wire length per unit area over the resistance
per unit length and hence `R ∝ a / (N*a^2)`, is fitted too, to show the sweep does not
reach it.

In [7]:
from scipy.optimize import least_squares

def _crit_pred(params, Ns, as_, c=None):
    lA, q, t = params[:3]
    cc = params[3] if c is None else c
    return lA + q * np.log10(as_) - t * np.log10(np.clip(Ns * as_**2 - cc, 1e-6, None))

def fit_critical(Ns, as_, lR, c=None):
    x = Ns * as_**2
    if c is None:
        r = least_squares(lambda pr: _crit_pred(pr, Ns, as_) - lR, [3.0, 0.0, 1.0, 2.0],
                          bounds=([-10, -10, 0, 0], [10, 10, 10, x.min() - 1e-3]))
        return r.x
    r = least_squares(lambda pr: _crit_pred(pr, Ns, as_, c) - lR, [3.0, 0.0, 1.0])
    return np.append(r.x, c)

def _r2(lR, pr):
    return 1.0 - np.sum((lR - pr) ** 2) / np.sum((lR - lR.mean()) ** 2)

# (a) threshold free, all trials
crit = fit_critical(Ns_flat, as_flat, logR)
r2_crit = _r2(logR, _crit_pred(crit, Ns_flat, as_flat))

# (b) threshold pinned at the infinite-system value, trials above it only
C_INF = 5.64
above = x_flat > C_INF
crit_inf = fit_critical(Ns_flat[above], as_flat[above], logR[above], C_INF)
r2_crit_inf = _r2(logR[above], _crit_pred(crit_inf, Ns_flat[above], as_flat[above]))
Xa = np.column_stack([np.ones(above.sum()), np.log10(Ns_flat[above]), np.log10(as_flat[above])])
r2_general_above = _r2(logR[above], Xa @ np.linalg.lstsq(Xa, logR[above], rcond=None)[0])

# (c) dense wire-limited limit, q = t = 1
lA_dense = np.mean(logR - np.log10(as_flat) + np.log10(x_flat))
r2_dense = _r2(logR, lA_dense + np.log10(as_flat) - np.log10(x_flat))

# cluster bootstrap over (N, a) cells for (a)
rng = np.random.default_rng(0)
cell_keys = list(merged)
boot = []
for _ in range(300):
    ks = [cell_keys[i] for i in rng.integers(0, len(cell_keys), len(cell_keys))]
    n_b, a_b, R_b = [], [], []
    for k in ks:
        v = merged[k]; n_b += [k[0]] * len(v); a_b += [k[1]] * len(v); R_b += v
    boot.append(fit_critical(np.array(n_b), np.array(a_b), np.log10(np.array(R_b))))
boot = np.array(boot)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5], axis=0)

fit_critical_form = {
    "model": "R_eff = A * a^q * (N*a^2 - c)^(-t), log10 least squares over individual trials",
    "n_trials": int(len(logR)),
    "A": float(10 ** crit[0]), "q": float(crit[1]), "t": float(crit[2]), "c": float(crit[3]),
    "ci95": {"q": [float(ci_lo[1]), float(ci_hi[1])], "t": [float(ci_lo[2]), float(ci_hi[2])],
             "c": [float(ci_lo[3]), float(ci_hi[3])]},
    "bootstrap": "300 resamples of the (N, a) cells",
    "r2": float(r2_crit),
    "c_pinned": {"c": C_INF, "n_trials": int(above.sum()), "A": float(10 ** crit_inf[0]),
                 "q": float(crit_inf[1]), "t": float(crit_inf[2]), "r2": float(r2_crit_inf),
                 "r2_general_same_trials": float(r2_general_above)},
    "dense_limit": {"model": "R_eff = A * a / (N*a^2)", "A": float(10 ** lA_dense), "r2": float(r2_dense)},
    "reference_t_2d": 1.31,
}
print(f"critical form, c free:   R = {10**crit[0]:.0f} * a^{crit[1]:.3f} * (N a^2 - {crit[3]:.2f})^-{crit[2]:.3f}   R^2 = {r2_crit:.3f}")
print(f"   95% CI (cell bootstrap): q [{ci_lo[1]:.2f}, {ci_hi[1]:.2f}]   t [{ci_lo[2]:.2f}, {ci_hi[2]:.2f}]   c [{ci_lo[3]:.2f}, {ci_hi[3]:.2f}]")
print(f"critical form, c = {C_INF} on the {above.sum()} trials above it:   q = {crit_inf[1]:.3f}   t = {crit_inf[2]:.3f}   R^2 = {r2_crit_inf:.3f}"
      f"   (general N^p a^q fit on the same trials: R^2 = {r2_general_above:.3f})")
print(f"dense wire-limited limit R = A a/(N a^2):   R^2 = {r2_dense:.3f}")
print(f"for comparison: general N^p a^q fit R^2 = {r2_general:.3f}, N a^2 collapse R^2 = {r2:.3f}; 2D conductivity exponent t = 1.31")

# collapse plot: R / a^q against (N a^2 - c)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
xs = x_flat - crit[3]
sc = ax.scatter(xs, R_flat / as_flat**crit[1], c=as_flat, cmap='plasma', s=18, alpha=0.6, edgecolor='none')
xl = np.logspace(np.log10(xs.min()), np.log10(xs.max()), 100)
ax.plot(xl, 10**crit[0] * xl**(-crit[2]), 'k--', linewidth=2,
        label=f'slope -t = -{crit[2]:.2f}  (R^2 = {r2_crit:.3f})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(f'N a^2 - c   (c = {crit[3]:.2f})'); ax.set_ylabel(f'R_eff / a^q   (q = {crit[1]:.2f})')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
fig.colorbar(sc, ax=ax, label='a')
plt.tight_layout()
plt.show()
_critical_fig = fig  # saved to the experiment directory once exp_manager_val exists (section 6)

critical form, c free:   R = 10511 * a^0.985 * (N a^2 - 2.26)^-1.445   R^2 = 0.920
   95% CI (cell bootstrap): q [0.87, 1.14]   t [1.37, 1.51]   c [1.79, 2.86]
critical form, c = 5.64 on the 2880 trials above it:   q = 1.050   t = 1.009   R^2 = 0.873   (general N^p a^q fit on the same trials: R^2 = 0.884)
dense wire-limited limit R = A a/(N a^2):   R^2 = 0.629
for comparison: general N^p a^q fit R^2 = 0.904, N a^2 collapse R^2 = 0.850; 2D conductivity exponent t = 1.31


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/2204118566.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Predicted-versus-measured for the critical form, in the same layout as the power-law
# comparison above (thesis Fig. "fit-comparison-critical"): the general law is repeated so
# the two can be read against each other. The residual structure of the general law (low
# at intermediate N a^2 for short sticks, high near the threshold) is what the inverse
# validation measures; the critical form does not share it.
pred_crit = _crit_pred(crit, Ns_flat, as_flat)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(logR, pred_general, s=10, alpha=0.4, label=f'general N^p a^q fit (R^2={r2_general:.3f})')
ax.scatter(logR, pred_crit, s=10, alpha=0.4, label=f'critical form, c free (R^2={r2_crit:.3f})')
lims = [logR.min() - 0.1, logR.max() + 0.1]
ax.plot(lims, lims, 'k--', alpha=0.4)
ax.set_xlabel('log10(measured R_eff)'); ax.set_ylabel('log10(predicted R_eff)')
ax.legend()
plt.tight_layout()
plt.show()
_comparison_crit_fig = fig


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/37101045.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(logR, pred, s=10, alpha=0.4, label=f'N*a^2 fit (R^2={r2:.3f})')
ax.scatter(logR, pred_general, s=10, alpha=0.4, label=f'general N^p a^q fit (R^2={r2_general:.3f})')
lims = [logR.min() - 0.1, logR.max() + 0.1]
ax.plot(lims, lims, 'k--', alpha=0.4)
ax.set_xlabel('log10(measured R_eff)'); ax.set_ylabel('log10(predicted R_eff)')
ax.legend()
plt.tight_layout()
plt.show()
_comparison_fig = fig


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/1147063207.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Invert: solution curve for a target R_eff

The fit only constrains the product $x = N a^2$, so for a target R_eff there's a whole curve of
valid (N, a) -- not a single point. We trace that curve for several representative targets, chosen
from the actual data range (5th/15th/30th/50th/70th/85th/95th percentile of the pooled mean
R_eff) so we're interpolating, not extrapolating far beyond what's been measured -- widened from
the original 3 percentiles (15/50/85) to cover the tails too, now that section 6's validation is
scaled up to match.

In [10]:
def x_for_target(R_target):
    """Required N*a^2 to hit a target R_eff, per the N*a^2 power law."""
    return 10 ** ((np.log10(R_target) - intercept) / slope)

def solution_curve_simple(R_target, a_values, N_bounds=(10, 400)):
    x_t = x_for_target(R_target)
    a_arr = np.asarray(a_values)
    N_arr = x_t / a_arr**2
    mask = (N_arr >= N_bounds[0]) & (N_arr <= N_bounds[1])
    return N_arr[mask], a_arr[mask]

def solution_curve_general(R_target, a_values, N_bounds=(10, 400)):
    a_arr = np.asarray(a_values)
    logN = (np.log10(R_target / Ap) - q_exp * np.log10(a_arr)) / p_exp
    N_arr = 10 ** logN
    mask = (N_arr >= N_bounds[0]) & (N_arr <= N_bounds[1])
    return N_arr[mask], a_arr[mask]

# two significant figures, not the nearest ten: with the clipped-rule pool the 5th and 15th
# percentiles (25.3 and 30.8 ohm) both rounded to 30, which duplicated four validation candidates
targets_demo = [float(f"{v:.2g}") for v in np.percentile(R_mean_pts, [5, 15, 30, 50, 70, 85, 95])]
assert len(set(targets_demo)) == len(targets_demo), targets_demo
print(f"Demo targets (5/15/30/50/70/85/95th percentile of measured R_eff): {targets_demo} ohm")

a_grid = np.linspace(0.05, 1.2, 300)
palette = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(8, 6.5))
sc = ax.scatter(Ns_pts, as_pts, c=np.log10(R_mean_pts), cmap='viridis_r', s=45,
                 edgecolor='k', linewidth=0.3, zorder=3, label='measured sweep points')
for i, R_t in enumerate(targets_demo):
    color = palette[i % len(palette)]
    N_simple, a_simple = solution_curve_simple(R_t, a_grid)
    N_general, a_general = solution_curve_general(R_t, a_grid)
    ax.plot(N_simple, a_simple, linewidth=2, linestyle='--', color=color,
            label=f'R_eff={R_t:.0f} ohm  (N*a^2 fit)')
    ax.plot(N_general, a_general, linewidth=2.5, linestyle='-', color=color,
            label=f'R_eff={R_t:.0f} ohm  (general fit)')
ax.set_xlabel('N'); ax.set_ylabel('a')
ax.set_title('Solution curves for a few target R_eff\n'
             f'(solid = general N^p a^q fit, dashed = N*a^2 fit; '
             f'inversion below uses the {"general" if use_general else "N*a^2"} one)')
ax.legend(fontsize=7, ncol=1, loc='upper right')
fig.colorbar(sc, ax=ax, label='log10(measured mean R_eff)')
plt.tight_layout()
plt.show()

Demo targets (5/15/30/50/70/85/95th percentile of measured R_eff): [26.0, 31.0, 40.0, 73.0, 160.0, 460.0, 890.0] ohm


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/3491568567.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Cross-check: nearest already-measured points

An independent sanity check on the fit -- for each target, which already-measured (N, a) points
came closest to it in practice?

In [11]:
def nearest_points(R_target, k=5):
    d = np.abs(np.log10(R_mean_pts) - np.log10(R_target))
    return np.argsort(d)[:k]

for R_t in targets_demo:
    print(f"\nNearest measured points to R_eff = {R_t:.0f} ohm:")
    for i in nearest_points(R_t):
        print(f"  N={Ns_pts[i]:4d}  a={as_pts[i]:.2f}   mean R_eff={R_mean_pts[i]:8.1f} ohm")


Nearest measured points to R_eff = 26 ohm:
  N=  65  a=1.10   mean R_eff=    26.0 ohm
  N=  95  a=0.80   mean R_eff=    25.9 ohm
  N=  75  a=1.00   mean R_eff=    26.3 ohm
  N=  80  a=0.90   mean R_eff=    26.4 ohm
  N= 130  a=0.60   mean R_eff=    25.6 ohm

Nearest measured points to R_eff = 31 ohm:
  N=  80  a=0.80   mean R_eff=    31.1 ohm
  N=  65  a=1.00   mean R_eff=    31.2 ohm
  N=  85  a=0.80   mean R_eff=    30.5 ohm
  N= 135  a=0.50   mean R_eff=    32.3 ohm
  N=  60  a=1.10   mean R_eff=    29.6 ohm

Nearest measured points to R_eff = 40 ohm:
  N=  50  a=1.00   mean R_eff=    40.5 ohm
  N=  95  a=0.60   mean R_eff=    39.3 ohm
  N=  50  a=1.10   mean R_eff=    39.1 ohm
  N= 150  a=0.40   mean R_eff=    38.8 ohm
  N=  55  a=0.90   mean R_eff=    42.2 ohm

Nearest measured points to R_eff = 73 ohm:
  N=  45  a=0.80   mean R_eff=    73.3 ohm
  N=  80  a=0.50   mean R_eff=    72.0 ohm
  N=  60  a=0.60   mean R_eff=    74.4 ohm
  N=  30  a=1.10   mean R_eff=    74.8 ohm
  N=  4

---
## 6. Validate the inversion with new experiments

For each demo target, pick four candidates off its solution curve, spanning compact to sparse
(`a` = 0.2, 0.4, 0.7, 1.0) -- widened from the original 2 (0.3/0.8) now that section 4 also
covers 7 targets instead of 3, so this checks the inversion across more of the (N, a) plane, not
just its middle. Node count is capped the same way as the extended forward sweep, so this stays
cheap per-network even if a solution curve would otherwise wander into the dense (high-N, high-a)
corner -- though with 7 targets x 4 candidates x `n_trials_val` trials each, the total network
count is now comparable to the forward sweep's own scale, so this section is the one worth
running on the cluster rather than locally.

In [12]:
def safe_candidate(N, a, node_cap=3000, N_floor=10):
    N = int(round(N))
    a = float(np.clip(a, 0.05, 1.2))
    nodes_est = 0.5 * N**2 * a**2
    if nodes_est > node_cap:
        N = int(np.sqrt(node_cap / (0.5 * a**2)))
    return max(N, N_floor), a

# Same rule as cluster/run_validation_task.py: a candidate is admitted only if the N the
# solution curve asks for is affordable (node budget) and physical (N floor). Replacing an
# unaffordable N by the largest affordable one, as an earlier version did, put the candidate
# off the curve and made one (N, a) -- same seed, same networks -- stand in for several
# targets. Those candidates are recorded as infeasible instead, and every admitted (N, a)
# is unique.
validation_plan, infeasible_candidates = [], []
for R_t in targets_demo:
    for a_c, tag in [(0.2, "very-compact"), (0.4, "compact-ish"), (0.7, "sparse-ish"), (1.0, "very-sparse")]:
        N_pred = predict_N(R_t, a_c)
        N_c, a_c2 = safe_candidate(N_pred, a_c)
        if N_c != int(round(N_pred)):
            infeasible_candidates.append({"R_target": R_t, "a": a_c2, "tag": tag, "N_curve": float(N_pred),
                                          "reason": "node budget" if 0.5 * N_pred**2 * a_c2**2 > 3000 else "N floor"})
            continue
        validation_plan.append({"R_target": R_t, "N": N_c, "a": a_c2, "tag": tag, "N_curve": float(N_pred)})
assert len({(v["N"], v["a"]) for v in validation_plan}) == len(validation_plan), "duplicate candidates"

print(f"{len(validation_plan)} validation candidates (predicted with the "
      f"{'general' if use_general else 'N*a^2'} fit), {len(infeasible_candidates)} infeasible:")
for v in validation_plan:
    print(f"  target={v['R_target']:7.1f} ohm   N={v['N']:4d}  a={v['a']:.2f}   ({v['tag']})")
for v in infeasible_candidates:
    print(f"  skipped target={v['R_target']:7.1f} ohm   a={v['a']:.2f}   ({v['tag']}): curve asks for N={v['N_curve']:.0f}, {v['reason']}")

22 validation candidates (predicted with the general fit), 6 infeasible:
  target=   26.0 ohm   N= 104  a=0.70   (sparse-ish)
  target=   26.0 ohm   N=  62  a=1.00   (very-sparse)
  target=   31.0 ohm   N=  93  a=0.70   (sparse-ish)
  target=   31.0 ohm   N=  56  a=1.00   (very-sparse)
  target=   40.0 ohm   N= 181  a=0.40   (compact-ish)
  target=   40.0 ohm   N=  80  a=0.70   (sparse-ish)
  target=   40.0 ohm   N=  48  a=1.00   (very-sparse)
  target=   73.0 ohm   N= 347  a=0.20   (very-compact)
  target=   73.0 ohm   N= 127  a=0.40   (compact-ish)
  target=   73.0 ohm   N=  56  a=0.70   (sparse-ish)
  target=   73.0 ohm   N=  34  a=1.00   (very-sparse)
  target=  160.0 ohm   N= 219  a=0.20   (very-compact)
  target=  160.0 ohm   N=  80  a=0.40   (compact-ish)
  target=  160.0 ohm   N=  36  a=0.70   (sparse-ish)
  target=  160.0 ohm   N=  21  a=1.00   (very-sparse)
  target=  460.0 ohm   N= 117  a=0.20   (very-compact)
  target=  460.0 ohm   N=  43  a=0.40   (compact-ish)
  target=  

In [13]:
n_trials_val = 30
max_attempts_val = 200

exp_manager_val = setup_experiment()
exp_manager_val.save_figure(_collapse_fig, "scaling_fit_collapse.png")
exp_manager_val.save_figure(_comparison_fig, "scaling_fit_comparison.png")
exp_manager_val.save_figure(_critical_fig, "scaling_fit_critical_collapse.png")
exp_manager_val.save_figure(_comparison_crit_fig, "scaling_fit_comparison_critical.png")


EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 884
  Timestamp: 20260916_081438
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_884

  [OK] Figure saved: scaling_fit_collapse.png


  [OK] Figure saved: scaling_fit_comparison.png
  [OK] Figure saved: scaling_fit_critical_collapse.png


  [OK] Figure saved: scaling_fit_comparison_critical.png


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_884/scaling_fit_comparison_critical.png'

In [14]:
cluster_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "cluster"))
cache_dir = os.path.join(cluster_dir, "validation_results")
cache_files = [os.path.join(cache_dir, f"candidate_{i:02d}.json") for i in range(len(validation_plan))]
use_cache = len(validation_plan) > 0 and all(os.path.exists(p) for p in cache_files)

validation_results = []

if use_cache:
    print(f"Found all {len(validation_plan)} candidate results cached in {cache_dir} "
          f"(from the parallel SLURM array run) -- loading instead of recomputing.")
    for p in cache_files:
        with open(p) as f:
            validation_results.append(json.load(f))
        v = validation_results[-1]
        mean_str = f"{v['achieved_mean_ohm']:8.1f}" if v['achieved_mean_ohm'] is not None else "     n/a"
        err_str = f"{v['relative_error'] * 100:+6.1f}%" if v['relative_error'] is not None else "   n/a"
        print(f"target={v['R_target']:7.1f} ohm   N={v['N']:4d} a={v['a']:.2f} ({v['tag']:13s})   "
              f"achieved mean={mean_str} ohm   rel.err={err_str}")
else:
    if os.path.isdir(cache_dir):
        print(f"Cache dir {cache_dir} exists but is incomplete "
              f"({sum(os.path.exists(p) for p in cache_files)}/{len(validation_plan)} candidates) -- recomputing all.")
    for plan in validation_plan:
        N, a, R_t = plan["N"], plan["a"], plan["R_target"]
        seed_base = stable_seed(N, round(a, 3), "inverse_validation", BOUNDARY_RULE)  # not hash(): see pipeline_utils.stable_seed
        np.random.seed(seed_base)

        R_effs = []
        total_attempts = 0
        while len(R_effs) < n_trials_val:
            res, attempts = generate_percolating_network(N, a, k_uniform=R0, max_attempts=max_attempts_val, boundary=BOUNDARY_RULE)
            total_attempts += attempts
            if res is None:
                print(f"  [WARNING] N={N} a={a}: gave up after {total_attempts} draws, "
                      f"only found {len(R_effs)}/{n_trials_val}")
                break
            rl = distance_dependent_impedance(res, R0)
            R_effs.append(rl['impedance'])

        achieved_mean = float(np.mean(R_effs)) if R_effs else None
        rel_err = (achieved_mean - R_t) / R_t if achieved_mean else None
        validation_results.append({
            "R_target": R_t, "N": N, "a": a, "tag": plan["tag"], "n": len(R_effs),
            "achieved_mean_ohm": achieved_mean,
            "achieved_median_ohm": float(np.median(R_effs)) if R_effs else None,
            "achieved_std_ohm": float(np.std(R_effs)) if R_effs else None,
            "relative_error": rel_err,
            "R_eff_values": R_effs,
        })
        mean_str = f"{achieved_mean:8.1f}" if achieved_mean is not None else "     n/a"
        err_str = f"{rel_err * 100:+6.1f}%" if rel_err is not None else "   n/a"
        print(f"target={R_t:7.1f} ohm   N={N:4d} a={a:.2f} ({plan['tag']:13s})   "
              f"achieved mean={mean_str} ohm   rel.err={err_str}")

Found all 22 candidate results cached in /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/cluster/validation_results (from the parallel SLURM array run) -- loading instead of recomputing.
target=   26.0 ohm   N= 104 a=0.70 (sparse-ish   )   achieved mean=    27.0 ohm   rel.err=  +3.9%
target=   26.0 ohm   N=  62 a=1.00 (very-sparse  )   achieved mean=    31.4 ohm   rel.err= +20.9%
target=   31.0 ohm   N=  94 a=0.70 (sparse-ish   )   achieved mean=    32.2 ohm   rel.err=  +3.9%
target=   31.0 ohm   N=  56 a=1.00 (very-sparse  )   achieved mean=    36.0 ohm   rel.err= +16.1%
target=   40.0 ohm   N= 182 a=0.40 (compact-ish  )   achieved mean=    31.1 ohm   rel.err= -22.3%
target=   40.0 ohm   N=  81 a=0.70 (sparse-ish   )   achieved mean=    38.5 ohm   rel.err=  -3.7%
target=   40.0 ohm   N=  48 a=1.00 (very-sparse  )   achieved mean=    42.5 ohm   rel.err=  +6.2%
target=   73.0 ohm   N= 3

In [15]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
targets_v = [v["R_target"] for v in validation_results if v["achieved_mean_ohm"] is not None]
achieved_v = [v["achieved_mean_ohm"] for v in validation_results if v["achieved_mean_ohm"] is not None]
tags_v = [v["tag"] for v in validation_results if v["achieved_mean_ohm"] is not None]

colors = {"very-compact": "tab:red", "compact-ish": "tab:blue",
          "sparse-ish": "tab:orange", "very-sparse": "tab:green"}
for tag in set(tags_v):
    xs = [t for t, tg in zip(targets_v, tags_v) if tg == tag]
    ys = [a for a, tg in zip(achieved_v, tags_v) if tg == tag]
    ax.scatter(xs, ys, s=80, label=tag, color=colors.get(tag), edgecolor='k', linewidth=0.5)

if targets_v and achieved_v:
    lims = [min(targets_v + achieved_v) * 0.7, max(targets_v + achieved_v) * 1.3]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='perfect inversion')
    ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('target R_eff (ohm)'); ax.set_ylabel('achieved mean R_eff (ohm)')
# no in-image title: the thesis caption states the law behind the candidates
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()
exp_manager_val.save_figure(fig, "inversion_validation_target_vs_achieved.png")

  [OK] Figure saved: inversion_validation_target_vs_achieved.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49612/3061567792.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_884/inversion_validation_target_vs_achieved.png'

---
## 7. Save results

In [16]:
summary = {
    "experiment": exp_manager_val.get_info(),
    "forward_data_sources": summary_paths,
    "n_pooled_points": n_pts,
    "n_pooled_trials": n_total_trials,
    "n_duplicate_cells_skipped": n_dup_cells,
    "fit_Na2": {"slope": float(slope), "intercept": float(intercept), "A": float(A), "r2": float(r2),
                "model": "R_eff = A * (N * a^2) ^ slope"},
    "fit_general": {"A": float(Ap), "p": float(p_exp), "q": float(q_exp), "r2": float(r2_general),
                     "model": "R_eff = A * N^p * a^q"},
    "fit_critical_form": fit_critical_form,
    "used_fit_for_inversion": "general" if use_general else "N*a^2",
    "targets_demo": targets_demo,
    "validation_plan": validation_plan,
    "infeasible_candidates": infeasible_candidates,
    "validation_results": validation_results,
}

exp_manager_val.save_json(summary, "inverse_problem_summary.json")

print(f"\nAll results saved to: {exp_manager_val.experiment_dir}")
for f in sorted(os.listdir(exp_manager_val.experiment_dir)):
    print(f"  - {f}")

  [OK] JSON saved: inverse_problem_summary.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_884
  - inverse_problem_summary.json
  - inversion_validation_target_vs_achieved.png
  - scaling_fit_collapse.png
  - scaling_fit_comparison.png
  - scaling_fit_comparison_critical.png
  - scaling_fit_critical_collapse.png


---
## Notes

- The N*a^2 collapse is not assumed blindly -- section 3.1 fits the general, unconstrained form
  $R_{eff} \approx A' N^p a^q$ and compares its R^2 and its $p$ vs $2q$ to the single-variable
  fit. The inversion in section 4 automatically uses whichever of the two fits the data actually
  supports better (see the printed `use_general` decision), rather than assuming the excluded-area
  variable that governs the *percolation threshold* also governs R_eff away from it.
- Either way, the inversion is fundamentally underdetermined: a target R_eff pins down a *curve*
  of (N, a) solutions, not a single pair (more, shorter sticks behave like fewer, longer ones).
  Picking one specific (N, a) off that curve requires an extra constraint from outside this model
  (e.g. a fabrication limit on how many sticks you can place, or on how long they can be).
- The validation experiments are an honest out-of-sample check: they redraw brand-new random
  networks at the candidate (N, a) rather than reusing anything from the fit, so the reported
  relative error reflects how well the chosen model actually predicts new draws (including their
  run-to-run variance), not just goodness-of-fit on the training data.
- Re-running this notebook after further forward sweeps (larger N, more (N, a) pairs, more
  trials per pair) will automatically pool the extra data in section 2 and sharpen both fits --
  nothing about the data loading is tied to one specific experiment folder.